# Notebook: 01_rt01_exploracao_inicial.ipynb
**id_rastreabilidade:** RTB_001  
**Versão:** v01  
**Data de criação:** "24/03/2026"  
**Última atualização:** "24/03/2026"  
**Responsável:** Ailton Vendramini

---

## Finalidade
Explorar a base CadÚnico `tudo.csv`, validar as colunas críticas do dicionário operacional e calcular preliminarmente a variável RT_01 do IVS-H (% de famílias com renda per capita menor ou igual a 1/2 salário mínimo).

---

## Base Conceitual
Documentos GitHub que definem o quê e o porquê:
- `dim_variavel_IVS_v01r7.md` → RT_01; Nota Metodológica; Resumo de Disponibilidade
- `conceito_vulnerabilidade_v03.md` → limites de cobertura do CadÚnico e uso analítico da vulnerabilidade
- `arquitetura_dados_IVS_IBGE_Horto_v10.md` → camada de dados e lógica geral do Atlas

> **⚠️ Fronteira IVS-H × IPST-H:** este notebook calcula variáveis do
> IVS-H — exatamente as 16 variáveis do IVS/IPEA (IU_01–IU_03, CH_01–CH_08,
> RT_01–RT_05). Variáveis de deslocamento pertencem ao IPST-H e não devem
> ser calculadas, armazenadas ou inseridas em `fato_variavel_ivs_loteamento`.
> Referência: `dim_variavel_IVS_v01r7.md` — Nota Metodológica.

## Base Lógica
Artefatos que definem a estrutura de dados:
- `schema_IVS_v05.sql` → leitura exploratória; futuras tabelas de staging e fato
- `nota_tecnica_fato_ivs_loteamento_v04.sql` → separação entre fato de variáveis e fato do índice
- `matriz_rastreabilidade_operacional_v02.md` → RTB_001

---

## Fontes de Dados
**Arquivo de entrada:** `/tmp/cecad/tudo.csv`  
**Versão do dado:** `cadunico_v7_2025_12`  
**Periodo de referência:** `CadÚnico dez/2025`  
**Dicionário utilizado:** `dicionario_cadunico_operacional_v02.md`  
**Banco SQLite:** `hortolandia.db`  
**⚠️ Risco de schema:** Alto — verificar dicionário antes de nova carga

---

## Tabelas
**Leitura:** `STG_CADUNICO_RAW`  
**Escrita:** —  

---

## Outputs Gerados
| Caminho completo | tipo_output | Pode commitar? |
| --- | --- | --- |
| `outputs/tabelas/ivs_variaveis.csv` | exploratorio | Não |

---

## Variáveis IVS-H Calculadas
- [x] RT_01 — % famílias renda per capita menor ou igual a 1/2 SM  
- [ ] RT_04 — % domicílios dependentes de idosos  
- [ ] CH_05 — % mães chefes sem fund. completo, filho menor de 15  
- [ ] CH_06 — Taxa de analfabetismo 15 anos ou mais  
- [ ] CH_07 — % crianças sem morador com fund. completo

In [15]:
import pandas as pd

df14 = pd.read_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_14_tribuna_liberal.csv")

df18 = pd.read_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_18_tribuna_liberal.csv")

In [16]:
df = pd.concat([df14, df18], ignore_index=True)
df[['data','item','dimensao_ivs','tipo_relacao_variavel','nivel_criticidade','tipo_evento','polaridade_evento']]

,data,item,dimensao_ivs,tipo_relacao_variavel,nivel_criticidade,tipo_evento,polaridade_evento
0,2026-03-14,Medidas protetivas,capital_humano,indireta,alta,indicador,negativo
1,2026-03-14,Homicídio em madeireira,capital_humano,contextual,alta,problema,negativo
2,2026-03-14,Acidente fatal com motocicleta,capital_humano,contextual,alta,problema,negativo
3,2026-03-18,Forum de Empregabilidade,renda_trabalho,indireta,media,politica_publica,positivo
4,2026-03-18,Embalixo 5x2,renda_trabalho,indireta,media,indicador,positivo
5,2026-03-18,Ministra das Mulheres Jardim Amanda,capital_humano,contextual,media,politica_publica,positivo
6,2026-03-18,Forum de Empregabilidade,renda_trabalho,indireta,media,politica_publica,positivo
7,2026-03-18,Embalixo 5x2,renda_trabalho,indireta,media,indicador,positivo
8,2026-03-18,Ministra das Mulheres Jardim Amanda,capital_humano,contextual,media,politica_publica,positivo


In [17]:
df['data'].value_counts()

data
2026-03-18    6
2026-03-14    3
Name: count, dtype: int64

In [18]:
df['tipo_evento'].value_counts()

tipo_evento
politica_publica    4
indicador           3
problema            2
Name: count, dtype: int64

In [19]:
df['polaridade_evento'].value_counts()

polaridade_evento
positivo    6
negativo    3
Name: count, dtype: int64

In [20]:
df['dimensao_ivs'].value_counts()

dimensao_ivs
capital_humano    5
renda_trabalho    4
Name: count, dtype: int64

In [21]:
peso = {'baixa':1, 'media':2, 'alta':3, 'alerta':4}

df['peso'] = df['nivel_criticidade'].map(peso)

df.groupby('dimensao_ivs')['peso'].sum().sort_values(ascending=False)

dimensao_ivs
capital_humano    13
renda_trabalho     8
Name: peso, dtype: int64

In [22]:
df['tipo_relacao_variavel'] = df['tipo_relacao_variavel'].replace({
    'proxy': 'indireta'
})

In [23]:
df['tipo_relacao_variavel'].value_counts()

tipo_relacao_variavel
indireta      5
contextual    4
Name: count, dtype: int64

In [24]:
df.to_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_14_tribuna_liberal.csv", index=False)

In [25]:
import pandas as pd
import os

# caminho da pasta
pasta = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

# lista todos os arquivos csv
arquivos = [f for f in os.listdir(pasta) if f.endswith('.csv')]

for arquivo in arquivos:
    caminho = os.path.join(pasta, arquivo)
    
    df = pd.read_csv(caminho)
    
    # só aplica se a coluna existir
    if 'tipo_relacao_variavel' in df.columns:
        df['tipo_relacao_variavel'] = df['tipo_relacao_variavel'].replace({
            'proxy': 'indireta'
        })
        
        df.to_csv(caminho, index=False)
        print(f"✔ Atualizado: {arquivo}")
    else:
        print(f"⚠ Ignorado (sem coluna): {arquivo}")

✔ Atualizado: 2026_03_14_tribuna_liberal.csv
✔ Atualizado: 2026_03_18_tribuna_liberal.csv
✔ Atualizado: 2026_03_21_tribuna_liberal.csv
✔ Atualizado: 2026_03_24_tribuna_liberal.csv
✔ Atualizado: 2026_03_25_tribuna_liberal.csv
✔ Atualizado: 2026_03_26_tribuna_liberal.csv
✔ Atualizado: 2026_03_27_tribuna_liberal.csv
✔ Atualizado: 2026_03_28_tribuna_liberal.csv
✔ Atualizado: 2026_03_29_tribuna_liberal.csv
✔ Atualizado: 2026_03_31_tribuna_liberal.csv
✔ Atualizado: 2026_04_01_tribuna_liberal.csv
✔ Atualizado: 2026_04_02_tribuna_liberal.csv
✔ Atualizado: 2026_04_03_tribuna_liberal.csv
✔ Atualizado: 2026_04_05_tribuna_liberal.csv
✔ Atualizado: 2026_04_07_tribuna_liberal.csv
✔ Atualizado: 2026_04_08_tribuna_liberal.csv
✔ Atualizado: 2026_04_09_tribuna_liberal.csv
✔ Atualizado: 2026_04_10_tribuna_liberal.csv
✔ Atualizado: 2026_04_11_tribuna_liberal.csv
⚠ Ignorado (sem coluna): corpus_resumo_periodos_v07.csv


In [26]:
df = df.drop(columns=['peso'])

KeyError: "['peso'] not found in axis"

In [27]:
df = df.drop(columns=['peso'], errors='ignore')

In [30]:
df.columns

Index(['data', 'item_evento', 'pagina', 'dimensao_ivs', 'ca3d_variavel',
       'tipo_relaaao', 'resumo', 'origem', 'municipio_impactado',
       'tipo_abrangencia', 'tipo_evento', 'gravidade', 'observaaao',
       'ca3d_lot', 'confianaa', 'polaridade_evento', 'origem\t\t\t\t\t\t'],
      dtype='object')

In [31]:
df14 = pd.read_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_14_tribuna_liberal.csv")

df14 = df14[df14['data'] == '2026-03-14']

In [32]:
df14

,fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,nivel_criticidade,resumo,origem,municipio_impactado,tipo_abrangencia,tipo_evento,observacao,cod_loteamento,nivel_confianca_loteamento,polaridade_evento,tipo_origem_evento,peso
0,Tribuna Liberal,2026-03-14,Medidas protetivas,s/n,capital_humano,NaN,indireta,alta,94 medidas protetivas em jan-fev/2026 maior vo...,Hortolandia,Hortolandia,local,indicador,Indicador de violência doméstica e fragilidade...,nao_aplicavel,nao_aplicavel,negativo,seguranca,3
1,Tribuna Liberal,2026-03-14,Homicídio em madeireira,s/n,capital_humano,NaN,contextual,alta,Homem é assassinado a tiros durante ataque a m...,Hortolandia,Hortolandia,local,problema,Evento de violência urbana associado à pressão...,nao_aplicavel,nao_aplicavel,negativo,seguranca,3
2,Tribuna Liberal,2026-03-14,Acidente fatal com motocicleta,s/n,capital_humano,NaN,contextual,alta,Adolescente morre após motocicleta bater contr...,Hortolandia,Hortolandia,local,problema,Evento relacionado a risco juvenil e possível ...,nao_aplicavel,nao_aplicavel,negativo,transito,3


In [33]:
df14.to_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_14_tribuna_liberal.csv", index=False)

In [34]:
df14 = pd.read_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_14_tribuna_liberal.csv")

# manter só dia 14
df14 = df14[df14['data'] == '2026-03-14']

# remover peso (se existir)
df14 = df14.drop(columns=['peso'], errors='ignore')

# garantir nomes corretos (se necessário)
df14.columns = [col.strip() for col in df14.columns]

# salvar
df14.to_csv(r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2026_03_14_tribuna_liberal.csv", index=False)

In [35]:
df14.columns
df14

,fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,nivel_criticidade,resumo,origem,municipio_impactado,tipo_abrangencia,tipo_evento,observacao,cod_loteamento,nivel_confianca_loteamento,polaridade_evento,tipo_origem_evento
0,Tribuna Liberal,2026-03-14,Medidas protetivas,s/n,capital_humano,NaN,indireta,alta,94 medidas protetivas em jan-fev/2026 maior vo...,Hortolandia,Hortolandia,local,indicador,Indicador de violência doméstica e fragilidade...,nao_aplicavel,nao_aplicavel,negativo,seguranca
1,Tribuna Liberal,2026-03-14,Homicídio em madeireira,s/n,capital_humano,NaN,contextual,alta,Homem é assassinado a tiros durante ataque a m...,Hortolandia,Hortolandia,local,problema,Evento de violência urbana associado à pressão...,nao_aplicavel,nao_aplicavel,negativo,seguranca
2,Tribuna Liberal,2026-03-14,Acidente fatal com motocicleta,s/n,capital_humano,NaN,contextual,alta,Adolescente morre após motocicleta bater contr...,Hortolandia,Hortolandia,local,problema,Evento relacionado a risco juvenil e possível ...,nao_aplicavel,nao_aplicavel,negativo,transito


In [1]:
# renomeação temporária
!rename 00_governanca\series_jornalisticas tmp_series

!git add .
!git commit -m "renomeacao temporaria para limpar encoding"
!git push

# voltar para nome correto
!rename 00_governanca\tmp_series series_jornalisticas

!git add .
!git commit -m "padronizacao final sem acento"
!git push

O sistema não pode encontrar o arquivo especificado.


[main d7306db] renomeacao temporaria para limpar encoding
 1 file changed, 47 insertions(+), 12 deletions(-)


To https://github.com/ailtonfv/Atlas-Social-de-Hortolandia.git
   291f23c..d7306db  main -> main
O sistema não pode encontrar o arquivo especificado.


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


Everything up-to-date


In [2]:
import pandas as pd
import glob

# Lista todos os arquivos
arquivos = glob.glob("2026_*.csv")

dfs = []

for a in arquivos:
    try:
        df = pd.read_csv(
            a,
            sep=",",
            encoding="utf-8",
            engine="python",
            on_bad_lines="skip"
        )

        # limpeza básica
        df.columns = [str(c).strip() for c in df.columns]

        # garante colunas mínimas
        if "dimensao_ivs" not in df.columns:
            print(f"IGNORADO (sem estrutura): {a}")
            continue

        dfs.append(df)

    except Exception as e:
        print(f"ERRO em {a}: {e}")

# consolidação
df_total = pd.concat(dfs, ignore_index=True, sort=False)

# remove evento político (Clodoaldo)
df_total = df_total[
    ~df_total["item"].str.contains("Clodoaldo", na=False)
]

# remove registros sem polaridade
df_total = df_total[
    df_total["polaridade_evento"].notna()
]

# =========================
# RESULTADOS FINAIS
# =========================

print("=== BASE FINAL ===")
print("Total de eventos:", len(df_total))

print("\nPolaridade:")
print(df_total["polaridade_evento"].value_counts())

print("\nDimensões:")
print(df_total["dimensao_ivs"].value_counts())

ValueError: No objects to concatenate

In [ ]:
import os

pasta = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\series_jornalisticas'

conteudo = """fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,resumo,localidade,tipo_evento,gravidade,observacao,cod_loteamento,nivel_confianca_loteamento,polaridade_evento,tipo_origem_evento
Tribuna Liberal,2026-04-03,Monte Sinai — 152 UH e saneamento 650 familias,1,infraestrutura_urbana,IU_01,direta,152 unidades habitacionais para moradores de areas de risco em parceria com CDHU; entrega prevista 2o semestre 2026; redes subterraneas de esgoto e aguas pluviais para 650 familias ja concluidas em parceria com Sabesp,Monte Sinai,infraestrutura,alta,Evento de infraestrutura mais substantivo registrado no corpus; 650 familias com saneamento regularizado comparavel com IU_01; 152 UH insumo para estimar cobertura de familias em area de risco — cruzamento futuro com CadUnico,nao_identificado,medio,positivo,infraestrutura
Tribuna Liberal,2026-04-03,Vereador Clodoaldo — pre-candidatura deputado estadual pauta empregabilidade,7,renda_trabalho,SMIDS_GOV,contextual,Partido Podemos confirma pre-candidatura de Clodoaldo Santos a deputado estadual 2026; trajetoria pautada em empregabilidade e inclusao social; serie historica votos: 1016 (2016) — 2481 (2020) — 2916 (2024),Hortolandia,institucional,media,Sinaliza que pauta de empregabilidade e inclusao de Hortolandia tem tracao eleitoral em 2026; cria janela de oportunidade institucional para o Atlas,nao_aplicavel,nao_aplicavel,positivo,institucional
"""

caminho = os.path.join(pasta, '2026_04_03_tribuna_liberal.csv')
with open(caminho, 'w', encoding='utf-8') as f:
    f.write(conteudo)

print(f'✅ Arquivo criado: {caminho}')

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Correção dos CSVs Diários — Atlas Social de Hortolândia\n",
    "\n",
    "Este notebook corrige três problemas em todos os arquivos diários da série jornalística:\n",
    "\n",
    "1. **`localidade`** — qualquer valor fora de `Hortolândia` ou `Regional` vira `Hortolândia`\n",
    "2. **`cod_loteamento`** — decimais (`351.0` → `351`) e células vazias → `nao_aplicavel`\n",
    "3. **`nivel_confianca_loteamento`** — células vazias → `nao_aplicavel`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import glob\n",
    "import os\n",
    "\n",
    "# Pasta dos arquivos diários\n",
    "pasta = r'C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\bd_externos\\series_jornalisticas'\n",
    "\n",
    "arquivos = sorted(glob.glob(os.path.join(pasta, '????_??_??_tribuna_liberal.csv')))\n",
    "\n",
    "print(f'Arquivos encontrados: {len(arquivos)}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "localidades_validas = ['Hortolândia', 'Regional']\n",
    "total_corrigidos = 0\n",
    "\n",
    "for caminho in arquivos:\n",
    "    nome = os.path.basename(caminho)\n",
    "    df = pd.read_csv(caminho)\n",
    "    modificado = False\n",
    "    problemas = []\n",
    "\n",
    "    # 1. Localidade inválida\n",
    "    col_loc = 'localidade'\n",
    "    if col_loc in df.columns:\n",
    "        mask_loc = ~df[col_loc].isin(localidades_validas)\n",
    "        if mask_loc.any():\n",
    "            valores = df.loc[mask_loc, col_loc].unique().tolist()\n",
    "            problemas.append(f'localidade inválida: {valores}')\n",
    "            df.loc[mask_loc, col_loc] = 'Hortolândia'\n",
    "            modificado = True\n",
    "\n",
    "    # 2. cod_loteamento com decimal ou vazio\n",
    "    col_cod = 'cod_loteamento'\n",
    "    if col_cod in df.columns:\n",
    "        def corrigir_cod(val):\n",
    "            v = str(val).strip()\n",
    "            if v == '' or v == 'nan':\n",
    "                return 'nao_aplicavel'\n",
    "            try:\n",
    "                f = float(v)\n",
    "                if f == int(f):\n",
    "                    return str(int(f))\n",
    "                return v\n",
    "            except:\n",
    "                return v\n",
    "\n",
    "        novos = df[col_cod].apply(corrigir_cod)\n",
    "        if not novos.equals(df[col_cod].astype(str)):\n",
    "            problemas.append('cod_loteamento com decimal ou vazio')\n",
    "            df[col_cod] = novos\n",
    "            modificado = True\n",
    "\n",
    "    # 3. nivel_confianca_loteamento vazio\n",
    "    col_conf = 'nivel_confianca_loteamento'\n",
    "    if col_conf in df.columns:\n",
    "        mask_conf = df[col_conf].isna() | (df[col_conf].astype(str).str.strip() == '')\n",
    "        if mask_conf.any():\n",
    "            problemas.append(f'nivel_confianca vazio: {mask_conf.sum()} linha(s)')\n",
    "            df.loc[mask_conf, col_conf] = 'nao_aplicavel'\n",
    "            modificado = True\n",
    "\n",
    "    # Salva se houve modificação\n",
    "    if modificado:\n",
    "        df.to_csv(caminho, index=False)\n",
    "        total_corrigidos += 1\n",
    "        print(f'[CORRIGIDO] {nome}')\n",
    "        for p in problemas:\n",
    "            print(f'  → {p}')\n",
    "    else:\n",
    "        print(f'[OK] {nome}')\n",
    "\n",
    "print('=' * 60)\n",
    "print(f'Total corrigidos: {total_corrigidos}')\n",
    "print(f'Já estavam ok   : {len(arquivos) - total_corrigidos}')\n",
    "print('Concluído.')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


In [ ]:
import urllib.request
import os

pasta = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'

base_url = 'https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governan%C3%A7a/series_jornalisticas/'

arquivos = [
    '2026_03_14_tribuna_liberal.csv',
    '2026_03_18_tribuna_liberal.csv',
    '2026_03_21_tribuna_liberal.csv',
    '2026_03_24_tribuna_liberal.csv',
    '2026_03_25_tribuna_liberal.csv',
    '2026_03_26_tribuna_liberal.csv',
    '2026_03_27_tribuna_liberal.csv',
    '2026_03_28_tribuna_liberal.csv',
    '2026_03_29_tribuna_liberal.csv',
    '2026_03_31_tribuna_liberal.csv',
    '2026_04_01_tribuna_liberal.csv',
    '2026_04_02_tribuna_liberal.csv',
    'corpus_resumo_periodos.csv',
]

for arq in arquivos:
    destino = os.path.join(pasta, arq)
    url = base_url + arq
    urllib.request.urlretrieve(url, destino)
    print(f'✅ {arq}')

print('\nPronto.')

In [ ]:
import pandas as pd

path = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv'

df = pd.read_csv(path, sep=',')

df.loc[df['periodo'] == '2026-03', 'eventos_por_dimensao'] = 'CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:2'

df.to_csv(path, sep=',', index=False)

print("Feito.")
print(df[['periodo','eventos_por_dimensao']])

In [ ]:
import pandas as pd

path = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv'

df = pd.read_csv(path, sep=';')
print(df.columns.tolist())
print(df.head())

In [ ]:
import pandas as pd

path = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv'

df = pd.read_csv(path, sep=';')

df.loc[df['periodo'] == '2026-03', 'eventos_por_dimensao'] = 'CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:2'

df.to_csv(path, sep=';', index=False)

print("Feito.")

In [ ]:
import pandas as pd
import os

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
nome_arquivo = '2026_04_02_tribuna_liberal.csv'
caminho_completo = os.path.join(caminho, nome_arquivo)

colunas = [
    'fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel',
    'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento',
    'gravidade', 'observacao', 'cod_loteamento', 'nivel_confianca_loteamento',
    'polaridade_evento', 'tipo_origem_evento'
]

registros = [
    ['Tribuna Liberal', '2026-04-02',
     'Viaduto Vila Real entra na reta final em Hortolandia',
     12, 'infraestrutura_urbana', 'SMIDS_EXT', 'contextual',
     'Pavimentacao da Rua Argolino de Moraes concluida; rotatória e via a serem liberadas nos proximos dias; entrega total prevista entre final de abril e inicio de maio; prefeito Zeze Gomes acompanha obra',
     'Hortolandia', 'infraestrutura', 'media',
     'Infraestrutura viaria com potencial de reorganizacao do fluxo urbano e valorizacao territorial; impacto indireto futuro sobre IU_01 e IU_02; RP_6',
     63, 'medio', 'positivo', 'infraestrutura'],

    ['Tribuna Liberal', '2026-04-02',
     'GM registra 14 casos Lei Maria da Penha em marco — Hortolandia',
     4, 'capital_humano', 'CH_05', 'direta',
     'Balanco GM marco 2026; 14 ocorrencias Lei Maria da Penha registradas; 3316 ligacoes recebidas; 1068 com deslocamento de viatura; 304 BOs lavrados',
     'Hortolandia', 'violencia', 'alta',
     'Dado administrativo oficial auditavel; volume mensal LMP permite construcao de serie temporal; base para monitoramento padrao CH_05 por periodo',
     'nao_aplicavel', 'nao_aplicavel', 'negativo', 'seguranca'],

    ['Tribuna Liberal', '2026-04-02',
     'GM registra 36 averiguacoes criminais e ocorrencias multiplas em marco — Hortolandia',
     4, 'multidimensional', 'multidimensional', 'contextual',
     'Balanco GM marco 2026; 36 averiguacoes de fatos criminosos incluindo roubos furtos agressoes tentativas de feminicidio e homicidio; 480 atendimentos perturbacao sossego; 42 brigas; 68 apoios a orgaos publicos',
     'Hortolandia', 'violencia', 'alta',
     'Dado agregado mensal auditavel; volume e recorrencia permitem saida do evento isolado para padrao mensal; interface CH e RT; referencia para serie temporal do Atlas',
     'nao_aplicavel', 'nao_aplicavel', 'negativo', 'seguranca'],

    ['Tribuna Liberal', '2026-04-02',
     'Audiencia Publica Plano Diretor — Hortolandia 07/04/2026',
     4, 'governanca', 'SMIDS_GOV', 'contextual',
     'Audiencia publica para discussao do PLC 11/2025 — Plano Diretor do Municipio de Hortolandia; realizada na Camara Municipal em 07/04 as 19h; transmissao ao vivo YouTube',
     'Hortolandia', 'institucional', 'media',
     'Instrumento estruturante que define a base territorial do Atlas — RPs e loteamentos; legitima politicamente o modelo; acompanhar desdobramentos para ajustes em dim_loteamento',
     'nao_aplicavel', 'nao_aplicavel', 'positivo', 'institucional'],
]

df_novo = pd.DataFrame(registros, columns=colunas)
df_novo.to_csv(caminho_completo, index=False, encoding='utf-8-sig', sep=',')

print(f"Arquivo salvo: {caminho_completo}")
print(f"Registros: {len(df_novo)}")
df_novo

In [ ]:
import pandas as pd
import os

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
nome_arquivo = 'corpus_resumo_periodos.csv'
caminho_completo = os.path.join(caminho, nome_arquivo)

colunas = [
    'periodo',
    'total_eventos',
    'eventos_negativos',
    'eventos_positivos',
    'eventos_por_loteamento',
    'eventos_por_dimensao',
    'observacao_sintese'
]

# Primeiro registro — março 2026
registros = [
    [
        '2026-03',
        34,
        26,
        8,
        'Jd. Conceicao: 4 | Jd. Amanda: 3 | Parque Santo Andre: 2',
        'CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:1 | EXT:1',
        'Concentracao de eventos CH no Jd. Conceicao com presenca simultanea de violencia domestica e vulnerabilidade juvenil; IU elevado reflete pressao urbana estrutural; primeiro registro institucional positivo via Selo FNAS sinaliza capacidade de resposta da rede'
    ]
]

df_resumo = pd.DataFrame(registros, columns=colunas)
df_resumo.to_csv(caminho_completo, index=False, encoding='utf-8-sig', sep=',')

print(f"Arquivo salvo: {caminho_completo}")
df_resumo

In [ ]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))
print(f"Arquivos encontrados: {len(arquivos)}")

# Mapeamento tipo_origem_evento a partir de tipo_evento
mapa_origem = {
    'violencia'      : 'seguranca',
    'vulnerabilidade': 'assistencial',
    'politica_publica': 'institucional',
    'infraestrutura' : 'infraestrutura',
    'institucional'  : 'institucional',
    'problema'       : 'seguranca',  # default conservador — revisar caso a caso
}

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=None, engine='python', encoding='utf-8-sig')
    alteracoes = []

    # 1. Adicionar tipo_origem_evento se ausente
    if 'tipo_origem_evento' not in df.columns:
        df['tipo_origem_evento'] = df['tipo_evento'].map(mapa_origem).fillna('a_revisar')
        alteracoes.append('tipo_origem_evento adicionado')

    # 2. SMIDS_EXT só onde não há mapeamento defensável para variável IVS
    # Linha 3 dos arquivos: violência de gênero → CH_05
    mask_genero = (
        df['codigo_variavel'].astype(str).str.strip() == 'SMIDS_EXT'
    ) & (
        df['resumo'].astype(str).str.contains('estupro|violencia de genero|agress', case=False, na=False)
    )
    if mask_genero.any():
        df.loc[mask_genero, 'codigo_variavel'] = 'CH_05'
        alteracoes.append('CH_05 corrigido em violência de gênero')

    # 3. a_revisar → contextual + governanca (emenda/CRAS/investimento)
    mask_revisar = (
        df['tipo_relacao_variavel'].astype(str).str.strip() == 'a_revisar'
    )
    if mask_revisar.any():
        df.loc[mask_revisar, 'tipo_relacao_variavel'] = 'contextual'
        df.loc[mask_revisar, 'codigo_variavel'] = 'governanca'
        alteracoes.append('a_revisar → contextual/governanca')

    df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')
    print(f"✓ {os.path.basename(arquivo)} — {', '.join(alteracoes) if alteracoes else 'sem alterações'}")

print("\nMigração concluída.")
print("⚠️  Revisar registros tipo_evento=problema com tipo_origem_evento=seguranca — inferência conservadora.")

In [ ]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))

# Palavras-chave que indicam origem econômica
keywords_economico = [
    'gasolina', 'combustivel', 'preco', 'inflacao', 'custo de vida',
    'reajuste', 'tarifa', 'salario', 'renda', 'aluguel', 'energia'
]

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=None, engine='python', encoding='utf-8-sig')
    
    if 'tipo_origem_evento' not in df.columns:
        continue

    mask_economico = (
        df['tipo_origem_evento'] == 'seguranca'
    ) & (
        df['resumo'].astype(str).str.contains(
            '|'.join(keywords_economico), case=False, na=False
        )
    )

    corrigidos = mask_economico.sum()
    if corrigidos > 0:
        df.loc[mask_economico, 'tipo_origem_evento'] = 'economico'
        df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')
        print(f"✓ {os.path.basename(arquivo)} — {corrigidos} registro(s) → economico")
    else:
        print(f"  {os.path.basename(arquivo)} — sem alterações")

print("\nCorreção concluída.")

In [ ]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'

arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))
print(f"Arquivos encontrados: {len(arquivos)}")

tipos_positivos = ['politica_publica', 'infraestrutura', 'institucional']

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=None, engine='python', encoding='utf-8-sig')
    
    if 'polaridade_evento' in df.columns:
        antes = (df['polaridade_evento'] == 'positivo').sum()
        df.loc[df['tipo_evento'].isin(tipos_positivos), 'polaridade_evento'] = 'positivo'
        depois = (df['polaridade_evento'] == 'positivo').sum()
        corrigidos = depois - antes
        print(f"✓ {os.path.basename(arquivo)} — {corrigidos} registro(s) corrigido(s)")
    else:
        print(f"⚠️  {os.path.basename(arquivo)} — coluna polaridade_evento ausente")
    
    df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')

print("\nCorreção concluída.")

In [ ]:
import pandas as pd
import os

caminho_csv = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
nome_arquivo = '2026_04_01_tribuna_liberal.csv'
caminho_completo = os.path.join(caminho_csv, nome_arquivo)

colunas = [
    'fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel',
    'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento',
    'gravidade', 'observacao', 'cod_loteamento', 'nivel_confianca_loteamento'
]

registros = [
    ['Tribuna Liberal', '2026-04-01',
     'Hortolandia conquista Selo FNAS 2025 na area de assistencia social',
     8, 'governanca', 'multidimensional', 'contextual_positivo',
     'Municipio conquista pela primeira vez o Selo FNAS 2025; rede SUAS atende 30 mil familias equivalente a 72 mil pessoas por meio de CRAS CREAS Centro Pop Casa de Passagem e Abrigo Institucional; reconhecimento federal por excelencia na gestao orcamentaria financeira contabil e regulatoria do Fundo Municipal de Assistencia Social',
     'Hortolandia', 'institucional', 'nao_aplicavel',
     'Evidencia de capacidade institucional instalada; suporte empirico para interpretacao IVS alto mais IPST baixo; gestao funcional pode estar absorvendo pressao territorial; primeiro selo conquistado pelo municipio; secretaria Maria dos Anjos citada',
     'nao_aplicavel', 'nao_aplicavel'],

    ['Tribuna Liberal', '2026-04-01',
     'GM apreende drogas em abrigo e mulher e detida em Hortolandia',
     8, 'capital_humano', 'CH_05', 'direta',
     'Mulher acolhida no Abrigo Esperancar com quatro filhos detida com 28 porcoes de cocaina e 15 de maconha; cao farejador Thor localizou entorpecentes sob movel no dormitorio feminino; Conselho Tutelar acionado para as criancas',
     'Hortolandia', 'vulnerabilidade', 'alta',
     'Criancas em situacao de risco dentro de equipamento de protecao social; interface CH_05 e vulnerabilidade infantil multipla; Jardim Conceicao RP_5; abrigo como loteamento de referencia',
     57, 'alto'],

    ['Tribuna Liberal', '2026-04-01',
     'Homem baleado e morto em madeireira no Jardim Amanda II',
     9, 'multidimensional', 'CH_01', 'indireta',
     'Eduardo Zani de 46 anos baleado na cabeca em madeireira no Jardim Amanda II; socorro realizado por populares em veiculo ate UPA Jardim Amanda; vitima nao resistiu antes de chegar a unidade',
     'Hortolandia', 'violencia', 'alta',
     'Homicidio em ambiente de trabalho informal; multidimensional CH mais RT indireto; ausencia de socorro institucional imediato; Jardim Amanda Quadra 1 RP_3',
     457, 'medio'],

    ['Tribuna Liberal', '2026-04-01',
     'Irmaos autuados apos sequencia de roubos de motos no Jardim Conceicao',
     9, 'capital_humano', 'CH_08', 'direta',
     'Dois irmaos detidos pela PM apos sequencia de roubos de motocicletas; adolescente entre os suspeitos apreendido; confessaram dois roubos incluindo Honda CB 500F e Kawasaki Versys',
     'Hortolandia', 'violencia', 'alta',
     'CH direto adolescente em conflito com a lei CH_08; RT contextual pressao socioeconomica indireta; Jardim Conceicao RP_5',
     57, 'medio'],
]

df_novo = pd.DataFrame(registros, columns=colunas)

# Verificar separador de um arquivo existente antes de salvar
df_ref = pd.read_csv(
    os.path.join(caminho_csv, '2026_03_31_tribuna_liberal.csv'),
    nrows=1, sep=None, engine='python'
)
sep_detectado = ','  # fallback

df_novo.to_csv(caminho_completo, index=False, encoding='utf-8-sig', sep=sep_detectado)

print(f"Arquivo salvo: {caminho_completo}")
print(f"Registros: {len(df_novo)}")
df_novo

In [ ]:
import pandas as pd
import os

# verificar onde o notebook está rodando
print("Diretório atual:", os.getcwd())

# caminho relativo correto
caminho = "../dados/tudo.csv"

# verificar se o arquivo existe
if os.path.exists(caminho):
    print("Arquivo encontrado!")
else:
    print("ERRO: arquivo NÃO encontrado!")

# leitura do CSV
df = pd.read_csv(
    caminho,
    sep=';',          # separador do CadÚnico
    encoding='utf-8',
    low_memory=False
)

print("Arquivo carregado com sucesso!")

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.head()

In [ ]:
unidades_assistencia = df[' d.nom_unidade_territorial_fam'].dropna().unique()

len(unidades_assistencia)

In [ ]:
df[' d.cod_familiar_fam'].nunique()

In [ ]:
df_unidades = pd.DataFrame(unidades_assistencia, columns=[' unidade_assistencia_bruta'])

df_unidades.to_csv("../outputs/unidades_assistencia_brutas.csv", index=False)

🔍 Achados Principais & Preliminares na análise do CadÚnico de Dezembro de 2025

A análise exploratória revelou que o campo não representa núcleos geográficos puro, mas sim uma estrutura híbrida, contendo:

🟢 Unidades de Proteção Social Básica
CRAS AMANDA
CRAS SANTA CLARA
CRAS NOVO ÂNGULO
etc.
🟡 Unidades de Proteção Social Especial
CREAS
🔴 Equipamentos específicos
CENTRO POP
🟠 Variações nominais e granularidade inconsistente
CRAS PRIMAVERA
CRAS PRIMAVERA CHICO VIGILANTE
CRAS ROSOLEN
CRAS ROSOLEN JOEL ALVES ASSUNCAO
⚠️ Diagnóstico Técnico

O campo d.nom_unidade_territorial_fam apresenta:

Mistura de unidade administrativa e território
Ausência de padronização textual
Inclusão de nomes institucionais e patronímicos
Variação de granularidade (macro vs microterritório)

👉 Portanto, trata-se de um:

Campo semântico híbrido (não normalizado)

📊 Evidência Quantitativa
Total de unidades distintas identificadas: 12
Número esperado de CRAS no município: 7

👉 Indica presença de:

duplicidade semântica
sobreposição de categorias
ruído estrutural no dado
🧠 Interpretação Conceitual

O campo representa, na prática:

Núcleo operacional da assistência social, e não núcleo geográfico formal.

Ou seja:

Reflete a lógica de atendimento do SUAS
Não corresponde diretamente a:
bairros
loteamentos
divisões do IBGE

In [ ]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

In [ ]:
print('unidade_raw' in df.columns)
print('unidade_norm' in df.columns)
print('unidade_oficial' in df.columns)

In [ ]:
df.columns = df.columns.str.strip()
for c in df.columns:
    print(repr(c))

In [ ]:
[col for col in df.columns if 'unidad' in c.lower() or 'equip' in c.lower() or 'cras' in c.lower()]

In [ ]:
df['p.ind_atend_cras_memb'].value_counts(dropna=False)

In [ ]:
for col in df.columns:
    if df[col].nunique() <= 20:
        print(col, df[col].nunique())

In [ ]:
df['d.nom_unidade_territorial_fam'].value_counts()

In [ ]:
[col for col in df.columns if 'cpf' in col.lower()]

In [ ]:
df['p.num_cpf_pessoa'].nunique()

In [ ]:
df['p.num_cpf_pessoa'].isna().sum()

In [ ]:
df['p.num_cpf_pessoa'].notna().sum()

In [ ]:
df[df.duplicated('p.num_cpf_pessoa', keep=False)] \
    .sort_values('p.num_cpf_pessoa') \
    .head(20)

In [ ]:
df['p.num_cpf_pessoa'] = df['p.num_cpf_pessoa'].astype(str).str.strip()

In [ ]:
df['p.num_cpf_pessoa'].value_counts().head(10)

In [ ]:
df['cpf_tratado'] = (
    df['p.num_cpf_pessoa']
    .str.replace('.0', '', regex=False)
)

In [ ]:
df['cpf_tratado'] = df['cpf_tratado'].replace('nan', None)

In [ ]:
df['cpf_tratado'] = df['cpf_tratado'].str.zfill(11)

In [ ]:
df['cpf_tratado'].value_counts().head(10)

In [ ]:
df.loc[df['cpf_tratado'] == '00000000000', 'cpf_tratado'] = None

In [ ]:
df.shape

In [ ]:
df.head(9)

In [ ]:
df.columns.tolist()

In [ ]:
df['d.cod_familiar_fam'].nunique()

In [ ]:
df.dtypes

In [ ]:
df['data_cadastro'] = pd.to_datetime(
    df['d.dat_cadastramento_fam'],
    dayfirst=True,
    errors='coerce'
)

In [ ]:
df['ano_cadastro'] = df['data_cadastro'].dt.year

In [ ]:
familias_por_ano = (
    df.groupby('ano_cadastro')['d.cod_familiar_fam']
      .nunique()
      .reset_index()
      .sort_values('ano_cadastro')
)

In [ ]:
print(familias_por_ano)

In [ ]:
df['d.nom_localidade_fam'].value_counts().head(30)

In [ ]:
familias_por_territorio = (
    df.groupby('d.nom_localidade_fam')['d.cod_familiar_fam']
      .nunique()
      .reset_index()
      .sort_values('d.cod_familiar_fam', ascending=False)
)

In [ ]:
print(familias_por_territorio.head(60))

In [ ]:
familias_por_territorio.head(30)

In [ ]:
df['territorio_simples'] = (
    df['d.nom_localidade_fam']
    .str.replace(' I$', '', regex=True)
    .str.replace(' II$', '', regex=True)
    .str.replace(' 1$', '', regex=True)
    .str.replace(' 2$', '', regex=True)
)

In [ ]:
familias_territorio_ano.sort_values(
    ['ano_cadastro', 'd.cod_familiar_fam'],
    ascending=[True, False]
).head(20)

In [ ]:
'familias_territorio_ano' in globals()

In [ ]:
familias_territorio_ano = (
    df.groupby(['ano_cadastro', 'territorio_simples'])['d.cod_familiar_fam']
      .nunique()
      .reset_index()
)

print(familias_territorio_ano.head(10))

In [ ]:
[var for var in globals() if 'territorio' in var or 'familias' in var]

In [ ]:
'familias_territorio_ano' in globals()

In [ ]:
[var for var in globals() if 'territorio' in var or 'familias' in var]

In [ ]:
import pandas as pd
import glob
import os

In [ ]:
pasta = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'

arquivos = glob.glob(pasta + r'\2026_*.csv')

print(f'Arquivos Encontrados: {len(arquivos)}')
for a in arquivos:
    print(os.path.basename(a))

In [ ]:
df = pd.concat(
    [pd.read_csv(f, sep=',', encoding='utf-8') for f in arquivos],
    ignore_index=True
)

print(f'Total de registros: {len(df)}')
print(f'Colunas: {list(df.columns)}')

Linha a linha:

pd.read_csv(f, sep=',', encoding='utf-8') → lê um arquivo CSV como tabela
[... for f in arquivos] → list comprehension: para cada arquivo da lista, lê e cria uma tabela — isso substitui um for com 3 linhas por 1 linha
pd.concat(...) → empilha todas as tabelas em uma só
ignore_index=True → reinicia a numeração das linhas de 1 até o total
len(df) → número de linhas
df.columns → nomes das colunas

In [ ]:
df.head()

In [ ]:
df['dimensao_ivs'].value_counts()

In [ ]:
df['tipo_relacao_variavel'].value_counts()

In [ ]:
df_diretos = df[df['tipo_relacao_variavel'] == 'direta']

print(f'Registros Diretos: {len(df_diretos)}')
df_diretos[['data', 'item', 'codigo_variavel', 'localidade']].head(10)

df[df['coluna'] == 'valor'] → filtra linhas onde a condição é verdadeira — equivale ao filtro do Excel
df_diretos[['data', 'item', ...]] → seleciona só algumas colunas para exibir

In [ ]:
df.columns.tolist()

In [ ]:
df['codigo_variavel'].value_counts()

In [ ]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=',', engine='python', encoding='utf-8-sig')
    
    # Identificar todas as colunas com 'observacao' no nome
    colunas_obs = [c for c in df.columns if 'observacao' in c.lower()]
    
    if len(colunas_obs) > 1:
        # Consolidar todas em uma única coluna 'observacao'
        df['observacao'] = df[colunas_obs].apply(
            lambda row: ' '.join([str(v) for v in row if pd.notna(v) and str(v).strip() != '']),
            axis=1
        )
        # Remover colunas fantasmas
        df = df.drop(columns=[c for c in colunas_obs if c != 'observacao'])
        print(f"✓ {os.path.basename(arquivo)} — colunas fantasmas removidas: {colunas_obs}")
    else:
        print(f"  {os.path.basename(arquivo)} — sem problema")
    
    df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')

print("\nLimpeza concluída.")

In [ ]:
df = pd.concat(
    [pd.read_csv(f, sep=',', encoding='utf-8') for f in arquivos],
    ignore_index=True
)

print(f'Total de registros: {len(df)}')
print(f'Colunas: {list(df.columns)}')

In [ ]:
df.columns.tolist()

In [ ]:
df['codigo_variavel'].value_counts()

In [ ]:
# Renomear colunas com variações de 'observacao'
novas_colunas = []

for c in df.columns:
    if 'observacao' in c:
        novas_colunas.append('observacao')
    else:
        novas_colunas.append(c)

df.columns = novas_colunas

In [ ]:
df.columns.tolist()

In [ ]:
df.columns.tolist()

In [ ]:
# Ver o conteúdo de cada coluna observacao
posicoes = [i for i, c in enumerate(df.columns) if c == 'observacao']
print(f"Posições encontradas: {posicoes}")

for pos in posicoes:
    print(f"\n--- Posição {pos} ---")
    valores = df.iloc[:, pos].dropna()
    print(f"Registros preenchidos: {len(valores)}")
    print(valores.unique())

In [ ]:
# Consolidar as 4 colunas observacao em uma só
posicoes = [i for i, c in enumerate(df.columns) if c == 'observacao']

# Pegar o primeiro valor não vazio entre as 4 colunas, linha a linha
df['observacao'] = df.iloc[:, posicoes].apply(
    lambda row: ' | '.join([str(v).strip() for v in row if pd.notna(v) and str(v).strip() != '']),
    axis=1
)

# Remover duplicatas — manter só a primeira ocorrência
df = df.loc[:, ~df.columns.duplicated()]

print(f"Colunas após limpeza: {len(df.columns)}")
print(df['observacao'].dropna().head(5))

In [ ]:
# Remover tabs residuais dentro do texto
df['observacao'] = df['observacao'].str.replace('\t', ' ', regex=False)

# Confirmar
print(df['observacao'].head(5))

In [ ]:
print(df['observacao'].iloc[0])
print('---')
print(df['observacao'].iloc[1])

In [ ]:
import re
df['observacao'] = df['observacao'].str.replace(r' +', ' ', regex=True)

print(df['observacao'].iloc[0])
print('---')
print(df['observacao'].iloc[1])

In [ ]:
r'C:\Users\ailto\atlas_social_projeto\...'

In [ ]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))

for arquivo in arquivos:
    df_arq = pd.read_csv(arquivo, sep=',', encoding='utf-8-sig')
    
    # Renomear variações de observacao
    novas_colunas = []
    for c in df_arq.columns:
        if 'observacao' in c:
            novas_colunas.append('observacao')
        else:
            novas_colunas.append(c)
    df_arq.columns = novas_colunas
    
    # Consolidar colunas observacao duplicadas
    posicoes = [i for i, c in enumerate(df_arq.columns) if c == 'observacao']
    if len(posicoes) > 1:
        df_arq['observacao'] = df_arq.iloc[:, posicoes].apply(
            lambda row: ' | '.join([str(v).strip() for v in row if pd.notna(v) and str(v).strip() != '']),
            axis=1
        )
        df_arq = df_arq.loc[:, ~df_arq.columns.duplicated()]
    
    # Limpar tabs e espaços duplos
    df_arq['observacao'] = df_arq['observacao'].str.replace('\t', ' ', regex=False)
    df_arq['observacao'] = df_arq['observacao'].str.replace(r' +', ' ', regex=True)
    
    df_arq.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')
    print(f"✓ {os.path.basename(arquivo)}")

print("\nPronto. Arquivos corrigidos na origem.")

In [ ]:
df = pd.concat(
    [pd.read_csv(f, sep=',', encoding='utf-8-sig') for f in arquivos],
    ignore_index=True
)

print(f"Total de registros: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")
print(df.columns.tolist())

In [ ]:
import pandas as pd
import glob

# Lista todos os arquivos
arquivos = glob.glob("2026_*.csv")

dfs = []

for a in arquivos:
    try:
        df = pd.read_csv(
            a,
            sep=",",
            encoding="utf-8",
            engine="python",
            on_bad_lines="skip"
        )

        # limpeza básica
        df.columns = [str(c).strip() for c in df.columns]

        # garante colunas mínimas
        if "dimensao_ivs" not in df.columns:
            print(f"IGNORADO (sem estrutura): {a}")
            continue

        dfs.append(df)

    except Exception as e:
        print(f"ERRO em {a}: {e}")

# consolidação
df_total = pd.concat(dfs, ignore_index=True, sort=False)

# remove evento político (Clodoaldo)
df_total = df_total[
    ~df_total["item"].str.contains("Clodoaldo", na=False)
]

# remove registros sem polaridade
df_total = df_total[
    df_total["polaridade_evento"].notna()
]

# =========================
# RESULTADOS FINAIS
# =========================

print("=== BASE FINAL ===")
print("Total de eventos:", len(df_total))

print("\nPolaridade:")
print(df_total["polaridade_evento"].value_counts())

print("\nDimensões:")
print(df_total["dimensao_ivs"].value_counts())

In [6]:
"""
migrar_corpus.py
Atlas Social de Hortolândia — Migração em lote do corpus jornalístico
----------------------------------------------------------------------
Gera:
  REPOSITORIO_LOCAL/NOME.csv    — sobrescreve o original já normalizado
  output/corpus_consolidado.csv — todos concatenados
  output/audit_report.csv       — relatório de auditoria

Como usar:
  1. Confirme o caminho em REPOSITORIO_LOCAL abaixo
  2. Adicione os URLs na lista ARQUIVOS
  3. Execute: main()  (no Jupyter) ou  python migrar_corpus.py

Padrões-alvo:
  Separador  : vírgula
  Schema     : v10.2 (20 colunas)
  Data       : YYYY-MM-DD (ISO 8601)
  id_evento  : TL_YYYYMMDD_NNN
"""

import csv
import re
import os
import urllib.request
from io import StringIO

# ─────────────────────────────────────────────────────────────
# CAMINHO DO REPOSITÓRIO LOCAL
# ─────────────────────────────────────────────────────────────
REPOSITORIO_LOCAL = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

# ─────────────────────────────────────────────────────────────
# LISTA DE ARQUIVOS — adicione URLs raw aqui, um por linha
# ─────────────────────────────────────────────────────────────
ARQUIVOS = [
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_27_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_28_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_29_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_31_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_03_03_tribuna_liberal.csv",
    # Adicione os demais abaixo:
]

# ─────────────────────────────────────────────────────────────
# SCHEMA ALVO v10.2
# ─────────────────────────────────────────────────────────────
COLUNAS_V10 = [
    "id_evento", "data_publicacao", "fonte", "titulo", "pagina",
    "municipio", "localidade", "tipo_camada", "dimensao_analitica",
    "tipo_relacao_variavel", "codigo_variavel", "nivel_criticidade",
    "observacao", "aproximacao_variavel", "nota_analista",
    "cod_loteamento", "nivel_confianca_loteamento",
    "papel_no_ciclo", "id_caso_pressao", "entra_ipst",
]

# ─────────────────────────────────────────────────────────────
# MAPEAMENTO SCHEMA ANTIGO → v10.2
# ─────────────────────────────────────────────────────────────
MAPA_ANTIGO = {
    "fonte":                      "fonte",
    "data":                       "data_publicacao",
    "item":                       "titulo",
    "pagina":                     "pagina",
    "dimensao_ivs":               "dimensao_analitica",
    "codigo_variavel":            "codigo_variavel",
    "tipo_relacao_variavel":      "tipo_relacao_variavel",
    "nivel_criticidade":          "nivel_criticidade",
    "resumo":                     "__nota",
    "origem":                     "municipio",
    "municipio_impactado":        "__drop",
    "tipo_abrangencia":           "__nota",
    "tipo_evento":                "__tipo_camada",
    "cod_loteamento":             "cod_loteamento",
    "nivel_confianca_loteamento": "nivel_confianca_loteamento",
    "polaridade_evento":          "__nota",
    "tipo_origem_evento":         "__nota",
    "observacao":                 "observacao",
    "nota_analista":              "nota_analista",
}

MAPA_TIPO_EVENTO = {
    "politica_publica": "GOVERNANCA",
    "problema":         "PRESSAO_SOCIAL",
    "indicador":        "IVS",
    "contexto":         "CONTEXTO",
}


def detectar_separador(texto):
    primeira = texto.split("\n")[0]
    return ";" if primeira.count(";") > primeira.count(",") else ","


def detectar_schema(colunas):
    cl = [c.lower().strip().lstrip("\ufeff") for c in colunas]
    if "tipo_camada" in cl:
        return "v10"
    if "tipo_evento" in cl or "dimensao_ivs" in cl:
        return "antigo"
    return "desconhecido"


def normalizar_data(valor, nome_arquivo):
    v = valor.strip().strip('"')
    aviso = None
    if re.match(r"\d{4}-\d{2}-\d{2}$", v):
        return v, aviso
    m = re.match(r"(\d{2})/(\d{2})/(\d{4})$", v)
    if m:
        d, mes, a = m.groups()
        ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
        if ref:
            a_r, m_r, d_r = ref.groups()
            if mes != m_r or a != a_r:
                aviso = f"DATA INCORRETA: '{v}' corrigida para {a_r}-{m_r}-{d_r}"
                return f"{a_r}-{m_r}-{d_r}", aviso
        return f"{a}-{mes}-{d}", aviso
    aviso = f"FORMATO NAO RECONHECIDO: '{v}'"
    return v, aviso


def normalizar_id(valor, nome_arquivo, n):
    ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
    if not ref:
        return valor or f"TL_UNKNOWN_{n:03d}"
    a, mes, d = ref.groups()
    prefixo = f"TL_{a}{mes}{d}_"
    v = str(valor).strip()
    if re.match(rf"TL_{a}{mes}{d}_\d+$", v):
        return v
    return f"{prefixo}{n:03d}"


def linha_v10_para_saida(linha):
    return {col: linha.get(col, "").strip() for col in COLUNAS_V10}


def linha_antigo_para_v10(linha, nome_arquivo, n):
    nova = {col: "" for col in COLUNAS_V10}
    notas_extra = []
    for col_orig, col_dest in MAPA_ANTIGO.items():
        val = linha.get(col_orig, "").strip()
        if not val:
            continue
        if col_dest == "__drop":
            continue
        elif col_dest == "__nota":
            notas_extra.append(f"{col_orig}={val}")
        elif col_dest == "__tipo_camada":
            nova["tipo_camada"] = MAPA_TIPO_EVENTO.get(val, f"REVISAR({val})")
        else:
            nova[col_dest] = val
    if not nova.get("localidade"):
        nova["localidade"] = "nao_identificado"
    if not nova.get("papel_no_ciclo"):
        nova["papel_no_ciclo"] = "REVISAR"
    if not nova.get("entra_ipst"):
        nova["entra_ipst"] = "REVISAR"
    if notas_extra:
        base = nova.get("nota_analista", "")
        sufixo = " | ".join(notas_extra)
        nova["nota_analista"] = f"{base} | {sufixo}" if base else sufixo
    return nova


def processar_arquivo(url):
    nome = url.split("/")[-1]
    meta = {
        "arquivo": nome, "schema_original": "", "separador_original": "",
        "n_linhas": 0, "status": "", "avisos": "",
    }
    try:
        with urllib.request.urlopen(url) as resp:
            texto = resp.read().decode("utf-8-sig")
    except Exception as e:
        meta["status"] = f"ERRO_DOWNLOAD: {e}"
        print(f"  x {nome}: {meta['status']}")
        return nome, [], meta

    sep    = detectar_separador(texto)
    reader = csv.DictReader(StringIO(texto), delimiter=sep)
    schema = detectar_schema(reader.fieldnames or [])
    meta["separador_original"] = sep
    meta["schema_original"]    = schema

    linhas_saida = []
    avisos = []

    for n, linha in enumerate(reader, start=1):
        linha = {k.strip().lstrip("\ufeff"): v for k, v in linha.items() if k is not None}
        nova  = (linha_v10_para_saida(linha) if schema == "v10"
                 else linha_antigo_para_v10(linha, nome, n))
        data_norm, aviso_data = normalizar_data(nova.get("data_publicacao", ""), nome)
        nova["data_publicacao"] = data_norm
        if aviso_data:
            avisos.append(f"L{n}: {aviso_data}")
        nova["id_evento"] = normalizar_id(nova.get("id_evento", ""), nome, n)
        linhas_saida.append(nova)

    meta["n_linhas"] = len(linhas_saida)
    meta["avisos"]   = " || ".join(avisos) if avisos else ""
    meta["status"]   = "AVISOS" if avisos else "OK"
    if schema == "antigo":
        meta["status"] += " | REVISAO_MANUAL(papel_no_ciclo, entra_ipst)"

    flag = "!" if (avisos or schema == "antigo") else "v"
    print(f"  [{flag}] {nome}  schema={schema}  sep={repr(sep)}  "
          f"linhas={len(linhas_saida)}  {meta['status']}")
    for av in avisos:
        print(f"      -> {av}")

    return nome, linhas_saida, meta


def salvar_csv(caminho, linhas):
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS_V10, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(linhas)


def main():
    os.makedirs("output", exist_ok=True)
    relatorio   = []
    consolidado = []

    # Verifica se a pasta do repositório existe
    if not os.path.isdir(REPOSITORIO_LOCAL):
        print(f"ATENCAO: pasta do repositorio nao encontrada:\n  {REPOSITORIO_LOCAL}")
        print("Verifique o caminho em REPOSITORIO_LOCAL no inicio do script.")
        print("Os arquivos serao salvos apenas em output/\n")
        repo_ok = False
    else:
        repo_ok = True

    print(f"Processando {len(ARQUIVOS)} arquivo(s)...\n")

    for url in ARQUIVOS:
        nome, linhas, meta = processar_arquivo(url)
        relatorio.append(meta)
        consolidado.extend(linhas)

        if linhas:
            # Sempre salva em output/ como backup
            salvar_csv(f"output/{nome}", linhas)

            # Sobrescreve no repositório local se a pasta existir
            if repo_ok:
                caminho_repo = os.path.join(REPOSITORIO_LOCAL, nome)
                salvar_csv(caminho_repo, linhas)
                print(f"      -> salvo em repositorio: {caminho_repo}")

    # Arquivo consolidado e relatório ficam em output/
    salvar_csv("output/corpus_consolidado.csv", consolidado)

    campos_rel = ["arquivo", "schema_original", "separador_original",
                  "n_linhas", "status", "avisos"]
    with open("output/audit_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos_rel, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(relatorio)

    n_erros   = sum(1 for r in relatorio if "ERRO"    in r["status"])
    n_revisao = sum(1 for r in relatorio if "REVISAO" in r["status"])
    n_avisos  = sum(1 for r in relatorio if "AVISOS"  in r["status"] and "REVISAO" not in r["status"])
    n_ok      = sum(1 for r in relatorio if r["status"] == "OK")

    print(f"\n{'─'*58}")
    print(f"  Arquivos processados : {len(ARQUIVOS)}")
    print(f"  Total de linhas      : {len(consolidado)}")
    print(f"  OK sem avisos        : {n_ok}")
    print(f"  Com avisos de data   : {n_avisos}")
    print(f"  Revisao manual       : {n_revisao}  <- schema antigo")
    print(f"  Erros de download    : {n_erros}")
    print(f"{'─'*58}")
    print(f"  output/corpus_consolidado.csv")
    print(f"  output/audit_report.csv")
    if repo_ok:
        print(f"  {REPOSITORIO_LOCAL}/<nome>.csv  (prontos para commit)")
    print(f"\n  Proximo passo: GitHub Desktop -> Commit -> Push")


if __name__ == "__main__":
    main()


Processando 6 arquivo(s)...

  [v] 2025_12_27_tribuna_liberal.csv  schema=v10  sep=';'  linhas=4  OK
      -> salvo em repositorio: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_27_tribuna_liberal.csv
  [v] 2025_12_28_tribuna_liberal.csv  schema=v10  sep=';'  linhas=4  OK
      -> salvo em repositorio: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_28_tribuna_liberal.csv
  [!] 2025_12_29_tribuna_liberal.csv  schema=v10  sep=';'  linhas=5  AVISOS
      -> L1: DATA INCORRETA: '29/01/2026' corrigida para 2025-12-29
      -> L2: DATA INCORRETA: '29/01/2026' corrigida para 2025-12-29
      -> L3: DATA INCORRETA: '29/01/2026' corrigida para 2025-12-29
      -> L4: DATA INCORRETA: '29/01/2026' corrigida para 2025-12-29
      -> L5: DATA INCORRETA: '29/01/2026' corrigida para 2025-12-29
      -> salvo em repositorio: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_29_tribuna_lib

In [7]:
"""
migrar_corpus.py  v2
Atlas Social de Hortolândia — Migração em lote do corpus jornalístico
----------------------------------------------------------------------
Schemas suportados:
  v10        : 20 colunas v10.2, sep=;  (dez/2025)
  v10_var    : variante jan/17 com edicao/codigo_evento, sep=,
  antigo     : 18 colunas schema antigo, sep=,

Saídas:
  REPOSITORIO_LOCAL/<nome>.csv  — sobrescreve original no repo local
  output/corpus_consolidado.csv — todos concatenados
  output/audit_report.csv       — relatório de auditoria

Uso no Jupyter: cole o script numa célula e chame main()
"""

import csv, re, os, urllib.request
from io import StringIO

# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÃO
# ─────────────────────────────────────────────────────────────
REPOSITORIO_LOCAL = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

ARQUIVOS = [
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_27_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_28_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_29_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_31_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_17_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_24_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_03_03_tribuna_liberal.csv",
    # Adicione os demais abaixo:
]

# ─────────────────────────────────────────────────────────────
# SCHEMA ALVO v10.2
# ─────────────────────────────────────────────────────────────
COLUNAS_V10 = [
    "id_evento","data_publicacao","fonte","titulo","pagina",
    "municipio","localidade","tipo_camada","dimensao_analitica",
    "tipo_relacao_variavel","codigo_variavel","nivel_criticidade",
    "observacao","aproximacao_variavel","nota_analista",
    "cod_loteamento","nivel_confianca_loteamento",
    "papel_no_ciclo","id_caso_pressao","entra_ipst",
]

# ─────────────────────────────────────────────────────────────
# MAPEAMENTO SCHEMA ANTIGO → v10.2
# ─────────────────────────────────────────────────────────────
MAPA_ANTIGO = {
    "fonte":                      "fonte",
    "data":                       "data_publicacao",
    "item":                       "titulo",
    "pagina":                     "pagina",
    "dimensao_ivs":               "dimensao_analitica",
    "codigo_variavel":            "codigo_variavel",
    "tipo_relacao_variavel":      "tipo_relacao_variavel",
    "nivel_criticidade":          "nivel_criticidade",
    "resumo":                     "__nota",
    "origem":                     "municipio",
    "municipio_impactado":        "__drop",
    "tipo_abrangencia":           "__nota",
    "tipo_evento":                "__tipo_camada",
    "cod_loteamento":             "cod_loteamento",
    "nivel_confianca_loteamento": "nivel_confianca_loteamento",
    "polaridade_evento":          "__nota",
    "tipo_origem_evento":         "__nota",
    "observacao":                 "observacao",
    "nota_analista":              "nota_analista",
}

MAPA_TIPO_EVENTO = {
    "politica_publica": "GOVERNANCA",
    "problema":         "PRESSAO_SOCIAL",
    "indicador":        "IVS",
    "contexto":         "CONTEXTO",
}

# ─────────────────────────────────────────────────────────────
# MAPEAMENTO SCHEMA v10_variante (jan/17) → v10.2
# ─────────────────────────────────────────────────────────────
MAPA_V10_VAR = {
    "codigo_evento":              "id_evento",
    "data_publicacao":            "data_publicacao",
    "edicao":                     "__nota",
    "titulo":                     "titulo",
    "pagina":                     "pagina",
    "municipio":                  "municipio",
    "localidade":                 "localidade",
    "tipo_camada":                "tipo_camada",
    "dimensao_analitica":         "dimensao_analitica",
    "tipo_relacao_variavel":      "tipo_relacao_variavel",
    "codigo_variavel":            "codigo_variavel",
    "nivel_criticidade":          "nivel_criticidade",
    "observacao":                 "observacao",
    "papel_no_ciclo":             "papel_no_ciclo",
    "id_caso_pressao":            "id_caso_pressao",
    "entra_ipst":                 "entra_ipst",
    "nota_analista":              "nota_analista",
}

# ─────────────────────────────────────────────────────────────
# FUNÇÕES AUXILIARES
# ─────────────────────────────────────────────────────────────

def detectar_separador(texto):
    primeira = texto.split("\n")[0]
    return ";" if primeira.count(";") > primeira.count(",") else ","


def detectar_schema(colunas):
    cl = [c.lower().strip().lstrip("\ufeff") for c in colunas]
    if "codigo_evento" in cl or "edicao" in cl:
        return "v10_var"
    if "tipo_camada" in cl:
        return "v10"
    if "tipo_evento" in cl or "dimensao_ivs" in cl:
        return "antigo"
    return "desconhecido"


def normalizar_data(valor, nome_arquivo):
    v = valor.strip().strip('"')
    aviso = None
    if re.match(r"\d{4}-\d{2}-\d{2}$", v):
        return v, aviso
    m = re.match(r"(\d{2})/(\d{2})/(\d{4})$", v)
    if m:
        d, mes, a = m.groups()
        ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
        if ref:
            a_r, m_r, d_r = ref.groups()
            if mes != m_r or a != a_r:
                aviso = f"DATA INCORRETA: '{v}' corrigida para {a_r}-{m_r}-{d_r}"
                return f"{a_r}-{m_r}-{d_r}", aviso
        return f"{a}-{mes}-{d}", aviso
    aviso = f"FORMATO NAO RECONHECIDO: '{v}'"
    return v, aviso


def normalizar_id(valor, nome_arquivo, n):
    ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
    if not ref:
        return valor or f"TL_UNKNOWN_{n:03d}"
    a, mes, d = ref.groups()
    v = str(valor).strip()
    if re.match(rf"TL_{a}{mes}{d}_\d+$", v):
        return v
    return f"TL_{a}{mes}{d}_{n:03d}"


def normalizar_entra_ipst(valor):
    """Normaliza 'sim - Condicao B', 'nao - R24' etc. para sim/nao/avaliar."""
    v = valor.strip().lower()
    if v.startswith("sim"):   return "sim"
    if v.startswith("nao") or v.startswith("não"): return "nao"
    if v.startswith("aval"):  return "avaliar"
    return valor  # mantém se não reconhecer


def linha_v10_para_saida(linha):
    nova = {col: linha.get(col, "").strip() for col in COLUNAS_V10}
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_v10var_para_v10(linha, nome_arquivo, n):
    nova = {col: "" for col in COLUNAS_V10}
    notas_extra = []
    for col_orig, col_dest in MAPA_V10_VAR.items():
        val = linha.get(col_orig, "").strip()
        if not val:
            continue
        if col_dest == "__nota":
            notas_extra.append(f"{col_orig}={val}")
        else:
            nova[col_dest] = val
    # Campos fixos ausentes no v10_var
    if not nova.get("fonte"):
        nova["fonte"] = "Tribuna Liberal"
    if not nova.get("cod_loteamento"):
        nova["cod_loteamento"] = "nao_identificado"
    if not nova.get("nivel_confianca_loteamento"):
        nova["nivel_confianca_loteamento"] = "baixo"
    if notas_extra:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + " | ".join(notas_extra)
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_antigo_para_v10(linha, nome_arquivo, n):
    nova = {col: "" for col in COLUNAS_V10}
    notas_extra = []
    for col_orig, col_dest in MAPA_ANTIGO.items():
        val = linha.get(col_orig, "").strip()
        if not val:
            continue
        if col_dest == "__drop":
            continue
        elif col_dest == "__nota":
            notas_extra.append(f"{col_orig}={val}")
        elif col_dest == "__tipo_camada":
            nova["tipo_camada"] = MAPA_TIPO_EVENTO.get(val, f"REVISAR({val})")
        else:
            nova[col_dest] = val
    if not nova.get("localidade"):
        nova["localidade"] = "nao_identificado"
    if not nova.get("papel_no_ciclo"):
        nova["papel_no_ciclo"] = "REVISAR"
    if not nova.get("entra_ipst"):
        nova["entra_ipst"] = "REVISAR"
    if notas_extra:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + " | ".join(notas_extra)
    return nova


# ─────────────────────────────────────────────────────────────
# PROCESSAMENTO
# ─────────────────────────────────────────────────────────────

def processar_arquivo(url):
    nome = url.split("/")[-1]
    meta = {"arquivo": nome, "schema_original": "", "separador_original": "",
            "n_linhas": 0, "status": "", "avisos": ""}
    try:
        with urllib.request.urlopen(url) as resp:
            texto = resp.read().decode("utf-8-sig")
    except Exception as e:
        meta["status"] = f"ERRO_DOWNLOAD: {e}"
        print(f"  x {nome}: {meta['status']}")
        return nome, [], meta

    sep    = detectar_separador(texto)
    reader = csv.DictReader(StringIO(texto), delimiter=sep)
    schema = detectar_schema(reader.fieldnames or [])
    meta["separador_original"] = sep
    meta["schema_original"]    = schema

    linhas_saida, avisos = [], []

    for n, linha in enumerate(reader, start=1):
        linha = {k.strip().lstrip("\ufeff"): v for k, v in linha.items() if k is not None}

        if   schema == "v10":     nova = linha_v10_para_saida(linha)
        elif schema == "v10_var": nova = linha_v10var_para_v10(linha, nome, n)
        else:                     nova = linha_antigo_para_v10(linha, nome, n)

        data_norm, aviso_data = normalizar_data(nova.get("data_publicacao", ""), nome)
        nova["data_publicacao"] = data_norm
        if aviso_data:
            avisos.append(f"L{n}: {aviso_data}")

        nova["id_evento"] = normalizar_id(nova.get("id_evento", ""), nome, n)
        linhas_saida.append(nova)

    meta["n_linhas"] = len(linhas_saida)
    meta["avisos"]   = " || ".join(avisos) if avisos else ""
    meta["status"]   = "AVISOS" if avisos else "OK"
    if schema in ("antigo", "v10_var"):
        meta["status"] += f" | SCHEMA_{schema.upper()}_MIGRADO"

    flag = "!" if (avisos or schema != "v10") else "v"
    print(f"  [{flag}] {nome}  schema={schema}  sep={repr(sep)}  "
          f"linhas={len(linhas_saida)}  {meta['status']}")
    for av in avisos:
        print(f"      -> {av}")

    return nome, linhas_saida, meta


def salvar_csv(caminho, linhas):
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS_V10, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(linhas)


def main():
    os.makedirs("output", exist_ok=True)
    repo_ok = os.path.isdir(REPOSITORIO_LOCAL)
    if not repo_ok:
        print(f"ATENCAO: pasta do repositorio nao encontrada:\n  {REPOSITORIO_LOCAL}\n"
              f"Arquivos serao salvos apenas em output/\n")

    print(f"Processando {len(ARQUIVOS)} arquivo(s)...\n")
    relatorio, consolidado = [], []

    for url in ARQUIVOS:
        nome, linhas, meta = processar_arquivo(url)
        relatorio.append(meta)
        consolidado.extend(linhas)
        if linhas:
            salvar_csv(f"output/{nome}", linhas)
            if repo_ok:
                caminho_repo = os.path.join(REPOSITORIO_LOCAL, nome)
                salvar_csv(caminho_repo, linhas)
                print(f"      -> repo: {caminho_repo}")

    salvar_csv("output/corpus_consolidado.csv", consolidado)

    campos_rel = ["arquivo","schema_original","separador_original","n_linhas","status","avisos"]
    with open("output/audit_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos_rel, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(relatorio)

    schemas = {}
    for r in relatorio:
        s = r["schema_original"]
        schemas[s] = schemas.get(s, 0) + 1

    print(f"\n{'─'*58}")
    print(f"  Arquivos processados : {len(ARQUIVOS)}")
    print(f"  Total de linhas      : {len(consolidado)}")
    for s, qtd in schemas.items():
        print(f"  Schema {s:<12}  : {qtd} arquivo(s)")
    n_erros = sum(1 for r in relatorio if "ERRO" in r["status"])
    n_avisos = sum(1 for r in relatorio if "AVISOS" in r["status"])
    print(f"  Com avisos de data   : {n_avisos}")
    print(f"  Erros de download    : {n_erros}")
    print(f"{'─'*58}")
    print(f"  output/corpus_consolidado.csv")
    print(f"  output/audit_report.csv")
    if repo_ok:
        print(f"\n  Proximo passo: GitHub Desktop -> Commit -> Push")


if __name__ == "__main__":
    main()


Processando 9 arquivo(s)...

  [v] 2025_12_27_tribuna_liberal.csv  schema=v10  sep=','  linhas=4  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_27_tribuna_liberal.csv
  [v] 2025_12_28_tribuna_liberal.csv  schema=v10  sep=','  linhas=4  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_28_tribuna_liberal.csv
  [v] 2025_12_29_tribuna_liberal.csv  schema=v10  sep=','  linhas=5  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_29_tribuna_liberal.csv
  [v] 2025_12_30_tribuna_liberal.csv  schema=v10  sep=','  linhas=2  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_30_tribuna_liberal.csv
  [v] 2025_12_31_tribuna_liberal.csv  schema=v10  sep=','  linhas=7  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_31_tribuna_lib

In [8]:
"""
migrar_corpus.py  v2
Atlas Social de Hortolândia — Migração em lote do corpus jornalístico
----------------------------------------------------------------------
Schemas suportados:
  v10        : 20 colunas v10.2, sep=;  (dez/2025)
  v10_var    : variante jan/17 com edicao/codigo_evento, sep=,
  antigo     : 18 colunas schema antigo, sep=,

Saídas:
  REPOSITORIO_LOCAL/<nome>.csv  — sobrescreve original no repo local
  output/corpus_consolidado.csv — todos concatenados
  output/audit_report.csv       — relatório de auditoria

Uso no Jupyter: cole o script numa célula e chame main()
"""

import csv, re, os, urllib.request
from io import StringIO

# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÃO
# ─────────────────────────────────────────────────────────────
REPOSITORIO_LOCAL = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

ARQUIVOS = [
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_27_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_28_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_29_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_31_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_17_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_24_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_03_01_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_03_03_tribuna_liberal.csv",
    # Adicione os demais abaixo:
]

# ─────────────────────────────────────────────────────────────
# SCHEMA ALVO v10.2
# ─────────────────────────────────────────────────────────────
COLUNAS_V10 = [
    "id_evento","data_publicacao","fonte","titulo","pagina",
    "municipio","localidade","tipo_camada","dimensao_analitica",
    "tipo_relacao_variavel","codigo_variavel","nivel_criticidade",
    "observacao","aproximacao_variavel","nota_analista",
    "cod_loteamento","nivel_confianca_loteamento",
    "papel_no_ciclo","id_caso_pressao","entra_ipst",
]

# ─────────────────────────────────────────────────────────────
# MAPEAMENTO SCHEMA ANTIGO → v10.2
# ─────────────────────────────────────────────────────────────
MAPA_ANTIGO = {
    "fonte":                      "fonte",
    "data":                       "data_publicacao",
    "item":                       "titulo",
    "pagina":                     "pagina",
    "dimensao_ivs":               "dimensao_analitica",
    "codigo_variavel":            "codigo_variavel",
    "tipo_relacao_variavel":      "tipo_relacao_variavel",
    "nivel_criticidade":          "nivel_criticidade",
    "resumo":                     "__nota",
    "origem":                     "municipio",
    "municipio_impactado":        "__drop",
    "tipo_abrangencia":           "__nota",
    "tipo_evento":                "__tipo_camada",
    "cod_loteamento":             "cod_loteamento",
    "nivel_confianca_loteamento": "nivel_confianca_loteamento",
    "polaridade_evento":          "__nota",
    "tipo_origem_evento":         "__nota",
    "observacao":                 "observacao",
    "nota_analista":              "nota_analista",
}

MAPA_TIPO_EVENTO = {
    "politica_publica": "GOVERNANCA",
    "problema":         "PRESSAO_SOCIAL",
    "indicador":        "IVS",
    "contexto":         "CONTEXTO",
}

# ─────────────────────────────────────────────────────────────
# MAPEAMENTO SCHEMA v10_variante (jan/17) → v10.2
# ─────────────────────────────────────────────────────────────
MAPA_V10_VAR = {
    "codigo_evento":              "id_evento",
    "data_publicacao":            "data_publicacao",
    "edicao":                     "__nota",
    "titulo":                     "titulo",
    "pagina":                     "pagina",
    "municipio":                  "municipio",
    "localidade":                 "localidade",
    "tipo_camada":                "tipo_camada",
    "dimensao_analitica":         "dimensao_analitica",
    "tipo_relacao_variavel":      "tipo_relacao_variavel",
    "codigo_variavel":            "codigo_variavel",
    "nivel_criticidade":          "nivel_criticidade",
    "observacao":                 "observacao",
    "papel_no_ciclo":             "papel_no_ciclo",
    "id_caso_pressao":            "id_caso_pressao",
    "entra_ipst":                 "entra_ipst",
    "nota_analista":              "nota_analista",
}

# ─────────────────────────────────────────────────────────────
# FUNÇÕES AUXILIARES
# ─────────────────────────────────────────────────────────────

def detectar_separador(texto):
    primeira = texto.split("\n")[0]
    return ";" if primeira.count(";") > primeira.count(",") else ","


def detectar_schema(colunas):
    cl = [c.lower().strip().lstrip("\ufeff") for c in colunas]
    if "codigo_evento" in cl or "edicao" in cl:
        return "v10_var"
    if "tipo_camada" in cl:
        return "v10"
    if "tipo_evento" in cl or "dimensao_ivs" in cl:
        return "antigo"
    return "desconhecido"


def normalizar_data(valor, nome_arquivo):
    v = valor.strip().strip('"')
    aviso = None
    if re.match(r"\d{4}-\d{2}-\d{2}$", v):
        return v, aviso
    m = re.match(r"(\d{2})/(\d{2})/(\d{4})$", v)
    if m:
        d, mes, a = m.groups()
        ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
        if ref:
            a_r, m_r, d_r = ref.groups()
            if mes != m_r or a != a_r:
                aviso = f"DATA INCORRETA: '{v}' corrigida para {a_r}-{m_r}-{d_r}"
                return f"{a_r}-{m_r}-{d_r}", aviso
        return f"{a}-{mes}-{d}", aviso
    aviso = f"FORMATO NAO RECONHECIDO: '{v}'"
    return v, aviso


def normalizar_id(valor, nome_arquivo, n):
    ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
    if not ref:
        return valor or f"TL_UNKNOWN_{n:03d}"
    a, mes, d = ref.groups()
    v = str(valor).strip()
    if re.match(rf"TL_{a}{mes}{d}_\d+$", v):
        return v
    return f"TL_{a}{mes}{d}_{n:03d}"


def normalizar_entra_ipst(valor):
    """Normaliza 'sim - Condicao B', 'nao - R24' etc. para sim/nao/avaliar."""
    v = valor.strip().lower()
    if v.startswith("sim"):   return "sim"
    if v.startswith("nao") or v.startswith("não"): return "nao"
    if v.startswith("aval"):  return "avaliar"
    return valor  # mantém se não reconhecer


def linha_v10_para_saida(linha):
    nova = {col: linha.get(col, "").strip() for col in COLUNAS_V10}
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_v10var_para_v10(linha, nome_arquivo, n):
    nova = {col: "" for col in COLUNAS_V10}
    notas_extra = []
    for col_orig, col_dest in MAPA_V10_VAR.items():
        val = linha.get(col_orig, "").strip()
        if not val:
            continue
        if col_dest == "__nota":
            notas_extra.append(f"{col_orig}={val}")
        else:
            nova[col_dest] = val
    # Campos fixos ausentes no v10_var
    if not nova.get("fonte"):
        nova["fonte"] = "Tribuna Liberal"
    if not nova.get("cod_loteamento"):
        nova["cod_loteamento"] = "nao_identificado"
    if not nova.get("nivel_confianca_loteamento"):
        nova["nivel_confianca_loteamento"] = "baixo"
    if notas_extra:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + " | ".join(notas_extra)
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_antigo_para_v10(linha, nome_arquivo, n):
    nova = {col: "" for col in COLUNAS_V10}
    notas_extra = []
    for col_orig, col_dest in MAPA_ANTIGO.items():
        val = linha.get(col_orig, "").strip()
        if not val:
            continue
        if col_dest == "__drop":
            continue
        elif col_dest == "__nota":
            notas_extra.append(f"{col_orig}={val}")
        elif col_dest == "__tipo_camada":
            nova["tipo_camada"] = MAPA_TIPO_EVENTO.get(val, f"REVISAR({val})")
        else:
            nova[col_dest] = val
    if not nova.get("localidade"):
        nova["localidade"] = "nao_identificado"
    if not nova.get("papel_no_ciclo"):
        nova["papel_no_ciclo"] = "REVISAR"
    if not nova.get("entra_ipst"):
        nova["entra_ipst"] = "REVISAR"
    if notas_extra:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + " | ".join(notas_extra)
    return nova


# ─────────────────────────────────────────────────────────────
# PROCESSAMENTO
# ─────────────────────────────────────────────────────────────

def processar_arquivo(url):
    nome = url.split("/")[-1]
    meta = {"arquivo": nome, "schema_original": "", "separador_original": "",
            "n_linhas": 0, "status": "", "avisos": ""}
    try:
        with urllib.request.urlopen(url) as resp:
            texto = resp.read().decode("utf-8-sig")
    except Exception as e:
        meta["status"] = f"ERRO_DOWNLOAD: {e}"
        print(f"  x {nome}: {meta['status']}")
        return nome, [], meta

    sep    = detectar_separador(texto)
    reader = csv.DictReader(StringIO(texto), delimiter=sep)
    schema = detectar_schema(reader.fieldnames or [])
    meta["separador_original"] = sep
    meta["schema_original"]    = schema

    linhas_saida, avisos = [], []

    for n, linha in enumerate(reader, start=1):
        linha = {k.strip().lstrip("\ufeff"): v for k, v in linha.items() if k is not None}

        if   schema == "v10":     nova = linha_v10_para_saida(linha)
        elif schema == "v10_var": nova = linha_v10var_para_v10(linha, nome, n)
        else:                     nova = linha_antigo_para_v10(linha, nome, n)

        data_norm, aviso_data = normalizar_data(nova.get("data_publicacao", ""), nome)
        nova["data_publicacao"] = data_norm
        if aviso_data:
            avisos.append(f"L{n}: {aviso_data}")

        nova["id_evento"] = normalizar_id(nova.get("id_evento", ""), nome, n)
        linhas_saida.append(nova)

    meta["n_linhas"] = len(linhas_saida)
    meta["avisos"]   = " || ".join(avisos) if avisos else ""
    meta["status"]   = "AVISOS" if avisos else "OK"
    if schema in ("antigo", "v10_var"):
        meta["status"] += f" | SCHEMA_{schema.upper()}_MIGRADO"

    flag = "!" if (avisos or schema != "v10") else "v"
    print(f"  [{flag}] {nome}  schema={schema}  sep={repr(sep)}  "
          f"linhas={len(linhas_saida)}  {meta['status']}")
    for av in avisos:
        print(f"      -> {av}")

    return nome, linhas_saida, meta


def salvar_csv(caminho, linhas):
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS_V10, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(linhas)


def main():
    os.makedirs("output", exist_ok=True)
    repo_ok = os.path.isdir(REPOSITORIO_LOCAL)
    if not repo_ok:
        print(f"ATENCAO: pasta do repositorio nao encontrada:\n  {REPOSITORIO_LOCAL}\n"
              f"Arquivos serao salvos apenas em output/\n")

    print(f"Processando {len(ARQUIVOS)} arquivo(s)...\n")
    relatorio, consolidado = [], []

    for url in ARQUIVOS:
        nome, linhas, meta = processar_arquivo(url)
        relatorio.append(meta)
        consolidado.extend(linhas)
        if linhas:
            salvar_csv(f"output/{nome}", linhas)
            if repo_ok:
                caminho_repo = os.path.join(REPOSITORIO_LOCAL, nome)
                salvar_csv(caminho_repo, linhas)
                print(f"      -> repo: {caminho_repo}")

    salvar_csv("output/corpus_consolidado.csv", consolidado)

    campos_rel = ["arquivo","schema_original","separador_original","n_linhas","status","avisos"]
    with open("output/audit_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos_rel, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(relatorio)

    schemas = {}
    for r in relatorio:
        s = r["schema_original"]
        schemas[s] = schemas.get(s, 0) + 1

    print(f"\n{'─'*58}")
    print(f"  Arquivos processados : {len(ARQUIVOS)}")
    print(f"  Total de linhas      : {len(consolidado)}")
    for s, qtd in schemas.items():
        print(f"  Schema {s:<12}  : {qtd} arquivo(s)")
    n_erros = sum(1 for r in relatorio if "ERRO" in r["status"])
    n_avisos = sum(1 for r in relatorio if "AVISOS" in r["status"])
    print(f"  Com avisos de data   : {n_avisos}")
    print(f"  Erros de download    : {n_erros}")
    print(f"{'─'*58}")
    print(f"  output/corpus_consolidado.csv")
    print(f"  output/audit_report.csv")
    if repo_ok:
        print(f"\n  Proximo passo: GitHub Desktop -> Commit -> Push")


if __name__ == "__main__":
    main()


Processando 10 arquivo(s)...

  [v] 2025_12_27_tribuna_liberal.csv  schema=v10  sep=','  linhas=4  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_27_tribuna_liberal.csv
  [v] 2025_12_28_tribuna_liberal.csv  schema=v10  sep=','  linhas=4  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_28_tribuna_liberal.csv
  [v] 2025_12_29_tribuna_liberal.csv  schema=v10  sep=','  linhas=5  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_29_tribuna_liberal.csv
  [v] 2025_12_30_tribuna_liberal.csv  schema=v10  sep=','  linhas=2  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_30_tribuna_liberal.csv
  [v] 2025_12_31_tribuna_liberal.csv  schema=v10  sep=','  linhas=7  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_31_tribuna_li

In [9]:
"""
migrar_corpus.py  v3
Atlas Social de Hortolândia — Migração em lote do corpus jornalístico
----------------------------------------------------------------------
Schemas suportados:
  v10           : 20 colunas v10.2, sep=;  (dez/2025)
  v10_var_full  : 22 colunas — v10.2 completo + edicao + codigo_evento (jan/03–10)
  v10_var_lite  : 17 colunas — sem aproximacao_variavel, cod_loteamento, nivel_confianca (jan/17, abr)
  antigo        : 18 colunas schema antigo, sep=,

Saídas:
  REPOSITORIO_LOCAL/<nome>.csv  — sobrescreve original no repo local
  output/corpus_consolidado.csv — todos concatenados
  output/audit_report.csv       — relatório de auditoria

Uso no Jupyter: cole numa célula e chame main()
"""

import csv, re, os, urllib.request
from io import StringIO

# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÃO
# ─────────────────────────────────────────────────────────────
REPOSITORIO_LOCAL = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

ARQUIVOS = [
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_27_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_28_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_29_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2025_12_31_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_03_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_04_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_07_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_08_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_09_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_10_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_17_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_24_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_30_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_01_31_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_03_01_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_03_03_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_04_24_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_04_25_tribuna_liberal.csv",
    "https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governanca/series_jornalisticas/2026_04_26_tribuna_liberal.csv",
    # Adicione os demais abaixo:
]

# ─────────────────────────────────────────────────────────────
# SCHEMA ALVO v10.2 — 20 colunas
# ─────────────────────────────────────────────────────────────
COLUNAS_V10 = [
    "id_evento","data_publicacao","fonte","titulo","pagina",
    "municipio","localidade","tipo_camada","dimensao_analitica",
    "tipo_relacao_variavel","codigo_variavel","nivel_criticidade",
    "observacao","aproximacao_variavel","nota_analista",
    "cod_loteamento","nivel_confianca_loteamento",
    "papel_no_ciclo","id_caso_pressao","entra_ipst",
]

# ─────────────────────────────────────────────────────────────
# MAPEAMENTO SCHEMA ANTIGO → v10.2
# ─────────────────────────────────────────────────────────────
MAPA_ANTIGO = {
    "fonte":                      "fonte",
    "data":                       "data_publicacao",
    "item":                       "titulo",
    "pagina":                     "pagina",
    "dimensao_ivs":               "dimensao_analitica",
    "codigo_variavel":            "codigo_variavel",
    "tipo_relacao_variavel":      "tipo_relacao_variavel",
    "nivel_criticidade":          "nivel_criticidade",
    "resumo":                     "__nota",
    "origem":                     "municipio",
    "municipio_impactado":        "__drop",
    "tipo_abrangencia":           "__nota",
    "tipo_evento":                "__tipo_camada",
    "cod_loteamento":             "cod_loteamento",
    "nivel_confianca_loteamento": "nivel_confianca_loteamento",
    "polaridade_evento":          "__nota",
    "tipo_origem_evento":         "__nota",
    "observacao":                 "observacao",
    "nota_analista":              "nota_analista",
}

MAPA_TIPO_EVENTO = {
    "politica_publica": "GOVERNANCA",
    "problema":         "PRESSAO_SOCIAL",
    "indicador":        "IVS",
    "contexto":         "CONTEXTO",
}

# ─────────────────────────────────────────────────────────────
# DETECÇÃO DE SCHEMA
# ─────────────────────────────────────────────────────────────

def detectar_separador(texto):
    primeira = texto.split("\n")[0]
    return ";" if primeira.count(";") > primeira.count(",") else ","


def detectar_schema(colunas):
    cl = [c.lower().strip().lstrip("\ufeff") for c in colunas]
    if "tipo_evento" in cl or "dimensao_ivs" in cl:
        return "antigo"
    if "codigo_evento" in cl or "edicao" in cl:
        # Distingue full (tem cod_loteamento separado) de lite
        if "cod_loteamento" in cl:
            return "v10_var_full"
        return "v10_var_lite"
    if "tipo_camada" in cl:
        return "v10"
    return "desconhecido"

# ─────────────────────────────────────────────────────────────
# NORMALIZAÇÃO
# ─────────────────────────────────────────────────────────────

def normalizar_data(valor, nome_arquivo):
    v = valor.strip().strip('"')
    aviso = None
    if re.match(r"\d{4}-\d{2}-\d{2}$", v):
        return v, aviso
    m = re.match(r"(\d{2})/(\d{2})/(\d{4})$", v)
    if m:
        d, mes, a = m.groups()
        ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
        if ref:
            a_r, m_r, d_r = ref.groups()
            if mes != m_r or a != a_r:
                aviso = f"DATA INCORRETA: '{v}' corrigida para {a_r}-{m_r}-{d_r}"
                return f"{a_r}-{m_r}-{d_r}", aviso
        return f"{a}-{mes}-{d}", aviso
    aviso = f"FORMATO NAO RECONHECIDO: '{v}'"
    return v, aviso


def normalizar_id(valor, nome_arquivo, n):
    ref = re.search(r"(\d{4})_(\d{2})_(\d{2})", nome_arquivo)
    if not ref:
        return valor or f"TL_UNKNOWN_{n:03d}"
    a, mes, d = ref.groups()
    v = str(valor).strip()
    if re.match(rf"TL_{a}{mes}{d}_\d+$", v):
        return v
    return f"TL_{a}{mes}{d}_{n:03d}"


def normalizar_entra_ipst(valor):
    v = valor.strip().lower()
    if v.startswith("sim"):  return "sim"
    if v.startswith("n"):    return "nao"
    if v.startswith("aval"): return "avaliar"
    return valor

# ─────────────────────────────────────────────────────────────
# CONVERSÕES POR SCHEMA
# ─────────────────────────────────────────────────────────────

def linha_v10_para_saida(linha):
    nova = {col: linha.get(col, "").strip() for col in COLUNAS_V10}
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_v10_var_full(linha, nome_arquivo, n):
    """v10_var_full: tem todos os campos v10.2 + edicao + codigo_evento."""
    nova = {col: "" for col in COLUNAS_V10}
    # Copia todos os campos v10.2 diretamente
    for col in COLUNAS_V10:
        if col == "id_evento":
            nova[col] = linha.get("codigo_evento", "").strip()
        else:
            nova[col] = linha.get(col, "").strip()
    # edicao vai para nota_analista
    edicao = linha.get("edicao", "").strip()
    if edicao:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + f"edicao={edicao}"
    # Fonte padrão se ausente
    if not nova.get("fonte"):
        nova["fonte"] = "Tribuna Liberal"
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_v10_var_lite(linha, nome_arquivo, n):
    """v10_var_lite: sem aproximacao_variavel, cod_loteamento, nivel_confianca."""
    nova = {col: "" for col in COLUNAS_V10}
    mapa = {
        "codigo_evento": "id_evento",
        "data_publicacao": "data_publicacao",
        "titulo": "titulo",
        "pagina": "pagina",
        "municipio": "municipio",
        "localidade": "localidade",
        "tipo_camada": "tipo_camada",
        "dimensao_analitica": "dimensao_analitica",
        "tipo_relacao_variavel": "tipo_relacao_variavel",
        "codigo_variavel": "codigo_variavel",
        "nivel_criticidade": "nivel_criticidade",
        "observacao": "observacao",
        "papel_no_ciclo": "papel_no_ciclo",
        "id_caso_pressao": "id_caso_pressao",
        "entra_ipst": "entra_ipst",
        "nota_analista": "nota_analista",
    }
    notas_extra = []
    for col_orig, col_dest in mapa.items():
        val = linha.get(col_orig, "").strip()
        if val:
            nova[col_dest] = val
    edicao = linha.get("edicao", "").strip()
    if edicao:
        notas_extra.append(f"edicao={edicao}")
    if not nova.get("fonte"):
        nova["fonte"] = "Tribuna Liberal"
    if not nova.get("cod_loteamento"):
        nova["cod_loteamento"] = "nao_identificado"
    if not nova.get("nivel_confianca_loteamento"):
        nova["nivel_confianca_loteamento"] = "baixo"
    if notas_extra:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + " | ".join(notas_extra)
    nova["entra_ipst"] = normalizar_entra_ipst(nova.get("entra_ipst", ""))
    return nova


def linha_antigo_para_v10(linha, nome_arquivo, n):
    nova = {col: "" for col in COLUNAS_V10}
    notas_extra = []
    for col_orig, col_dest in MAPA_ANTIGO.items():
        val = linha.get(col_orig, "").strip()
        if not val:
            continue
        if col_dest == "__drop":
            continue
        elif col_dest == "__nota":
            notas_extra.append(f"{col_orig}={val}")
        elif col_dest == "__tipo_camada":
            nova["tipo_camada"] = MAPA_TIPO_EVENTO.get(val, f"REVISAR({val})")
        else:
            nova[col_dest] = val
    if not nova.get("localidade"):
        nova["localidade"] = "nao_identificado"
    if not nova.get("papel_no_ciclo"):
        nova["papel_no_ciclo"] = "REVISAR"
    if not nova.get("entra_ipst"):
        nova["entra_ipst"] = "REVISAR"
    if notas_extra:
        base = nova.get("nota_analista", "")
        nova["nota_analista"] = (base + " | " if base else "") + " | ".join(notas_extra)
    return nova

# ─────────────────────────────────────────────────────────────
# PROCESSAMENTO
# ─────────────────────────────────────────────────────────────

def processar_arquivo(url):
    nome = url.split("/")[-1]
    meta = {"arquivo": nome, "schema_original": "", "separador_original": "",
            "n_linhas": 0, "status": "", "avisos": ""}
    try:
        with urllib.request.urlopen(url) as resp:
            texto = resp.read().decode("utf-8-sig")
    except Exception as e:
        meta["status"] = f"ERRO_DOWNLOAD: {e}"
        print(f"  x {nome}: {meta['status']}")
        return nome, [], meta

    sep    = detectar_separador(texto)
    reader = csv.DictReader(StringIO(texto), delimiter=sep)
    schema = detectar_schema(reader.fieldnames or [])
    meta["separador_original"] = sep
    meta["schema_original"]    = schema

    linhas_saida, avisos = [], []

    for n, linha in enumerate(reader, start=1):
        linha = {k.strip().lstrip("\ufeff"): v for k, v in linha.items() if k is not None}

        if   schema == "v10":           nova = linha_v10_para_saida(linha)
        elif schema == "v10_var_full":  nova = linha_v10_var_full(linha, nome, n)
        elif schema == "v10_var_lite":  nova = linha_v10_var_lite(linha, nome, n)
        else:                           nova = linha_antigo_para_v10(linha, nome, n)

        data_norm, aviso_data = normalizar_data(nova.get("data_publicacao", ""), nome)
        nova["data_publicacao"] = data_norm
        if aviso_data:
            avisos.append(f"L{n}: {aviso_data}")

        nova["id_evento"] = normalizar_id(nova.get("id_evento", ""), nome, n)
        linhas_saida.append(nova)

    meta["n_linhas"] = len(linhas_saida)
    meta["avisos"]   = " || ".join(avisos) if avisos else ""
    meta["status"]   = "AVISOS" if avisos else "OK"
    if schema == "antigo":
        meta["status"] += " | REVISAO_MANUAL(papel_no_ciclo, entra_ipst)"

    flag = "!" if (avisos or schema == "antigo") else "v"
    print(f"  [{flag}] {nome}  schema={schema}  sep={repr(sep)}  "
          f"linhas={len(linhas_saida)}  {meta['status']}")
    for av in avisos:
        print(f"      -> {av}")

    return nome, linhas_saida, meta


def salvar_csv(caminho, linhas):
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS_V10, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(linhas)


def main():
    os.makedirs("output", exist_ok=True)
    repo_ok = os.path.isdir(REPOSITORIO_LOCAL)
    if not repo_ok:
        print(f"ATENCAO: pasta do repositorio nao encontrada:\n  {REPOSITORIO_LOCAL}\n"
              f"Arquivos serao salvos apenas em output/\n")

    print(f"Processando {len(ARQUIVOS)} arquivo(s)...\n")
    relatorio, consolidado = [], []

    for url in ARQUIVOS:
        nome, linhas, meta = processar_arquivo(url)
        relatorio.append(meta)
        consolidado.extend(linhas)
        if linhas:
            salvar_csv(f"output/{nome}", linhas)
            if repo_ok:
                caminho_repo = os.path.join(REPOSITORIO_LOCAL, nome)
                salvar_csv(caminho_repo, linhas)
                print(f"      -> repo: {caminho_repo}")

    salvar_csv("output/corpus_consolidado.csv", consolidado)

    campos_rel = ["arquivo","schema_original","separador_original","n_linhas","status","avisos"]
    with open("output/audit_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos_rel, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(relatorio)

    schemas = {}
    for r in relatorio:
        s = r["schema_original"]
        schemas[s] = schemas.get(s, 0) + 1

    print(f"\n{'─'*62}")
    print(f"  Arquivos processados : {len(ARQUIVOS)}")
    print(f"  Total de linhas      : {len(consolidado)}")
    for s, qtd in sorted(schemas.items()):
        print(f"  Schema {s:<18}: {qtd} arquivo(s)")
    n_erros  = sum(1 for r in relatorio if "ERRO"   in r["status"])
    n_avisos = sum(1 for r in relatorio if "AVISOS" in r["status"])
    n_rev    = sum(1 for r in relatorio if "REVISAO" in r["status"])
    print(f"  Com avisos de data   : {n_avisos}")
    print(f"  Revisao manual       : {n_rev}  <- schema antigo")
    print(f"  Erros de download    : {n_erros}")
    print(f"{'─'*62}")
    print(f"  output/corpus_consolidado.csv")
    print(f"  output/audit_report.csv")
    if repo_ok:
        print(f"\n  Proximo passo: GitHub Desktop -> Commit -> Push")


if __name__ == "__main__":
    main()


Processando 20 arquivo(s)...

  [v] 2025_12_27_tribuna_liberal.csv  schema=v10  sep=','  linhas=4  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_27_tribuna_liberal.csv
  [v] 2025_12_28_tribuna_liberal.csv  schema=v10  sep=','  linhas=4  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_28_tribuna_liberal.csv
  [v] 2025_12_29_tribuna_liberal.csv  schema=v10  sep=','  linhas=5  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_29_tribuna_liberal.csv
  [v] 2025_12_30_tribuna_liberal.csv  schema=v10  sep=','  linhas=2  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_30_tribuna_liberal.csv
  [v] 2025_12_31_tribuna_liberal.csv  schema=v10  sep=','  linhas=7  OK
      -> repo: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas\2025_12_31_tribuna_li